<a href="https://colab.research.google.com/github/toxzak-svg/time/blob/main/01_reversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Temporal Reversion Task

Tests whether systems correctly handle facts that revert to a previous state.

Example: CEO is Alice (2019), then Bob (2021), then Alice again (2023). 

In [ ]:
# @title ## Setup: Install Dependencies
!pip install sentence-transformers scikit-learn -q
print("✓ Dependencies installed")

In [ ]:
# @title ## Setup: Clone Repository
!git clone https://github.com/toxzak-svg/time.git /content/time
%cd /content/time
print("✓ Repository cloned")

In [ ]:
# @title ## Setup: Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title ## Load TemporalBench Dataset
import json
from pathlib import Path

data_dir = Path("/content/time/benchmarks")

# Load v1 facts and questions
facts_path = data_dir / "temporalbench_v1_facts.jsonl"
questions_path = data_dir / "temporalbench_v1_questions.jsonl"

facts = []
with open(facts_path) as f:
    for line in f:
        facts.append(json.loads(line))

questions = []
with open(questions_path) as f:
    for line in f:
        questions.append(json.loads(line))

print(f"Loaded {len(facts)} facts, {len(questions)} questions")

# Filter to Reversion task family
reversion_tasks = [q for q in questions if q.get("task_family") == "Reversion"]
print(f"Reversion tasks: {len(reversion_tasks)}")

# Show an example
if reversion_tasks:
    print("\nExample question:")
    print(json.dumps(reversion_tasks[0], indent=2))

In [ ]:
# @title ## Define TTA Systems (A–D + Baseline)

def system_a_baseline(question: dict, facts: list[dict]) -> str:
    """Retrieves all facts for the subject, returns the one with latest as_of_day."""
    subject = question["subject"]
    as_of = question["as_of_day"]
    domain = question.get("domain", "")
    
    matching = []
    for f in facts:
        if f["subject"] == subject and f["domain"] == domain:
            if f["t_valid_from"] <= as_of <= f["t_valid_until"]:
                matching.append(f)
    
    if matching:
        # Return latest fact (most recent t_valid_until)
        return max(matching, key=lambda x: x["t_valid_until"])
    return None

def system_b_latest(question: dict, facts: list[dict]) -> str:
    """Returns the most recently valid fact, ignoring validity windows."""
    subject = question["subject"]
    domain = question.get("domain", "")
    
    matching = [f for f in facts if f["subject"] == subject and f["domain"] == domain]
    if not matching:
        return None
    
    # Pick latest by t_valid_from
    return max(matching, key=lambda x: x["t_valid_from"])

def system_c_retrieve_all(question: dict, facts: list[dict]) -> str:
    """Retrieves all facts for subject (no filtering), relies on LLM to reason."""
    subject = question["subject"]
    domain = question.get("domain", "")
    matching = [f for f in facts if f["subject"] == subject and f["domain"] == domain]
    return matching  # Return all for LLM to reason

def system_d_validity_filter(question: dict, facts: list[dict]) -> str:
    """Filters by temporal validity interval, picks latest matching the as_of_day."""
    subject = question["subject"]
    as_of = question["as_of_day"]
    domain = question.get("domain", "")
    
    valid_facts = []
    for f in facts:
        if f["subject"] == subject and f["domain"] == domain:
            if f["t_valid_from"] <= as_of <= f["t_valid_until"]:
                valid_facts.append(f)
    
    if not valid_facts:
        return None
    
    # Return the one that is most recently valid (highest t_valid_until)
    return max(valid_facts, key=lambda x: x["t_valid_until"])

print("✓ Systems defined")

In [ ]:
# @title ## Run Benchmark on Reversion Tasks
import csv
from collections import defaultdict

results = {
    "A_baseline": [],
    "B_latest": [],
    "C_retrieve_all": [],
    "D_validity_filter": [],
}

for q in reversion_tasks:
    # System A
    result_a = system_a_baseline(q, facts)
    if result_a:
        correct = result_a["content"] == q["answer"]
        results["A_baseline"].append({"question_id": q["question_id"], "correct": correct, "predicted": result_a["content"], "expected": q["answer"]})
    
    # System B
    result_b = system_b_latest(q, facts)
    if result_b:
        correct = result_b["content"] == q["answer"]
        results["B_latest"].append({"question_id": q["question_id"], "correct": correct, "predicted": result_b["content"], "expected": q["answer"]})
    
    # System D
    result_d = system_d_validity_filter(q, facts)
    if result_d:
        correct = result_d["content"] == q["answer"]
        results["D_validity_filter"].append({"question_id": q["question_id"], "correct": correct, "predicted": result_d["content"], "expected": q["answer"]})

print("Results:")
for system, runs in results.items():
    if runs:
        acc = sum(1 for r in runs if r["correct"]) / len(runs) * 100
        print(f"  {system}: {acc:.1f}% ({len(runs)} tasks)")

In [ ]:
# @title ## Save Results
import pandas as pd

rows = []
for system, runs in results.items():
    for r in runs:
        rows.append({
            "system": system,
            "question_id": r["question_id"],
            "correct": r["correct"],
            "predicted": r["predicted"],
            "expected": r["expected"],
        })

df = pd.DataFrame(rows)
df.to_csv("/content/reversion_results.csv", index=False)
print(f"✓ Saved {len(df)} rows to /content/reversion_results.csv")
print(df.head())

## Results Summary

If System D (validity filter) performs poorly on Reversion tasks, it means
the system is not correctly handling reversion events where a fact returns
to a previous state. This is the core failure mode this task exposes.